**Monte Carlo integration of the standard normal PDF.**
The **standard normal probability density function** is:

$$f(x) = \frac{1}{\sqrt{2\pi}}\,e^{-x^2/2}$$

This notebook estimates the probability that a standard normal random variable
falls within one standard deviation of the mean - the famous **68-95-99.7 rule**:

$$P(-1 \leq Z \leq 1) = \int_{-1}^{1} f(x)\,dx \approx 0.6827$$

Unlike the parabola, this integral has no closed form in elementary functions.
It can be expressed using the error function,
$P(-1 \leq Z \leq 1) = \operatorname{erf}(1/\sqrt{2})$,
but that itself requires numerical evaluation.
This makes it a natural candidate for Monte Carlo integration.

The bounding rectangle spans $x \in [-1, 1]$ and $y \in [0, 0.5]$,
giving a sample area of $2 \times 0.5 = 1$.
The box top ($y = 0.5$) is deliberately above the PDF's maximum
value of $f(0) = 1/\sqrt{2\pi} \approx 0.3989$.
Because the sample area is exactly 1, the estimation formula simplifies to:

$$\hat{I} = 1 \times \frac{\text{dots under curve}}{\text{dots total}}$$

**Halton sequence (Numba-compiled).**
Same implementation as the previous Monte Carlo notebooks.

**Sampling region.**
The rectangle $[-1,1] \times [0, 0.5]$ encloses the region under the PDF
between $x = -1$ and $x = 1$.
Its area of exactly 1 makes the estimation formula particularly clean.

In [ ]:
"""mc_std_normal.ipynb"""

# Cell 01 - Halton quasi-random sequence, bounding rectangle, and sample count

%matplotlib inline

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.patches import Rectangle
from numba import float64, int64, vectorize
from scipy.special import erf


@vectorize([float64(int64, int64)], nopython=True)
def halton(n, p):
    h, f = 0, 1
    while n > 0:
        f = f / p
        h += (n % p) * f
        n = int(n / p)
    return h


bbox = Rectangle((-1, 0), 2, 0.5).get_bbox()  # x in [-1,1], y in [0, 0.5]
print(bbox)
print(f"Sample area = {bbox.width * bbox.height:.1f}   (est = 1 * dots_in / total)")

total_dots = 30_000
print(f"{total_dots = :,}")

**Generating the Halton sample points.**
Base 2 for $x$, base 3 for $y$, scaled to the bounding rectangle.

In [ ]:
# Cell 02 - Generate Halton sample points over the bounding rectangle
# Halton is indexed from n = 1; n = 0 would map to the corner of the rectangle
# Unlike the circle/sphere notebooks the y-box here is not symmetric about the
# region of interest, so the (1 - h) reflection changes the sample set; keep it

n_arr = np.arange(1, total_dots + 1, dtype=np.int64)

x = (1 - halton(n_arr, np.full(total_dots, 2, dtype=np.int64))) * bbox.width + bbox.x0
y = (1 - halton(n_arr, np.full(total_dots, 3, dtype=np.int64))) * bbox.height + bbox.y0

pd.DataFrame({"x": x[:5], "y": y[:5]})

**Standard normal PDF and signed distance.**
$d = y - f(x)$: negative means the point is under the curve,
positive means above it.

In [ ]:
# Cell 03 - Standard normal PDF and signed distance d = y - f(x)
# d <= 0: point is under or on the PDF curve
# d >  0: point is above the PDF curve


def f(x):
    """Standard normal probability density function."""
    return 1.0 / np.sqrt(2.0 * np.pi) * np.exp(-(x**2) / 2.0)


d = y - f(x)

pd.DataFrame({"x": x[:5], "y": y[:5], "d": d[:5]})

**Classifying points as under or above the PDF curve.**

In [ ]:
# Cell 04 - Partition points: under/on PDF (d <= 0) vs above (d > 0)

x_in = x[d <= 0.0]
y_in = y[d <= 0.0]
x_out = x[d > 0.0]
y_out = y[d > 0.0]

pd.DataFrame(
    {"x_in": x_in[:5], "y_in": y_in[:5], "x_out": x_out[:5], "y_out": y_out[:5]}
)

**Scatter plot with PDF overlay.**
The full PDF curve is drawn across $[-4, 4]$ in green to show its
bell shape, while the Monte Carlo samples are confined to $[-1, 1]$.
Red points are under the curve; blue points are above it.

In [ ]:
# Cell 05 - Scatter plot with full PDF curve overlaid

x_curve = np.linspace(-4, 4, 300)
y_curve = f(x_curve)

plt.figure(figsize=(9, 6))
plt.scatter(x_in, y_in, color="red", marker=".", s=0.5)
plt.scatter(x_out, y_out, color="blue", marker=".", s=0.5)
plt.plot(
    x_curve,
    y_curve,
    color="green",
    lw=1.5,
    label=r"$f(x)=\frac{1}{\sqrt{2\pi}}\,e^{-x^2/2}$",
)
plt.axhline(0, color="gray", lw=0.8)
plt.axvline(0, color="gray", lw=0.8)
plt.xlim(-4.0, 4.0)
plt.ylim(-0.05, 0.55)
plt.title(
    "Monte Carlo Integration: Standard Normal PDF\n"
    r"Estimating $P(-1 \leq Z \leq 1)$"
)
plt.xlabel("x")
plt.ylabel("PDF")
plt.legend(loc="upper right", fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

**Estimating $P(-1 \leq Z \leq 1)$ and measuring the error.**
The exact value is $\operatorname{erf}(1/\sqrt{2}) \approx 0.682689492$,
computed here via `scipy.special.erf` for full precision.

In [ ]:
# Cell 06 - Compute the integral estimate and error
# Exact value: P(-1<=Z<=1) = erf(1/sqrt(2))

act = erf(1 / np.sqrt(2))  # exact: scipy.special.erf
est = (bbox.width * bbox.height) * np.count_nonzero(d <= 0) / total_dots
err = np.abs((est - act) / act)

print(f"dots total        : {total_dots:,}")
print(f"dots under curve  : {np.count_nonzero(d <= 0):,}")
print(f"actual  erf(1/√2) : {act:.9f}")
print(f"estimated         : {est:.6f}")
print(f"abs pct rel error : {err:.5%}")